# Resume DeepLab consistency · seed42

**Cho thành viên nhận bàn giao: chỉ cần input Cityscapes, bật GPU/Internet và Save & Run All.** Notebook tự tải checkpoint đã kiểm hash từ GitHub Release, ghép các phần và tiếp tục consistency từ epoch5–20. Baseline/aug20 được giữ nguyên và bỏ qua.

Cityscapes phải giữ version/mount `/kaggle/input/datasets/kavithak1388/cityscapes/Cityscape`, có ảnh và gtFine train/val. Gói Release giữ weights/optimizer/RNG/log/config/source, loại mask GT Cityscapes; notebook tạo lại3475mask từ input của bạn và kiểm hash từng bản đồ nhãn. Không cần ACDC hoặc code ZIP riêng.

Nếu đã thêm ZIP kết quả DeepLab gốc (hoặc thư mục đã giải nén), notebook ưu tiên input đó, không tải Release. Có nhiều bản DeepLab thì đặt RESUME_INPUT theo Copy path. B2 không được chọn làm DeepLab. Thiếu kết quả và không tải được Release thì báo lỗi, không train mới.

Giữ nguyên config/LR/seed và tổng20epoch. Timing T4 trước đó: còn16epoch khoảng3,26giờ train+validation, cộng setup/download/ZIP/checkpoint. Tải ZIP DeepLab mới khi `suite_complete`,3/3nhánh20/20.


In [ ]:
"""Standard-library source bootstrap shared by the two DG model notebooks."""
from pathlib import Path, PurePosixPath
import shutil
import stat
import zipfile


MARKERS=('scripts/run_dg_suite.py','src/training/checkpoints.py','src/models/registry.py')


def valid_source(path):
    path=Path(path)
    if path.is_dir():return all((path/n).is_file() for n in MARKERS)
    if path.is_file() and zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:return set(MARKERS).issubset(z.namelist())
    return False


def resolve_source(hint,input_root):
    hint=Path(hint);input_root=Path(input_root)
    if valid_source(hint):return hint
    search_root=hint if hint.is_dir() and hint.resolve()!=input_root.resolve() else input_root
    anchor=PurePosixPath(MARKERS[0])
    folders=sorted({p.parents[len(anchor.parts)-1] for p in search_root.rglob(anchor.name) if valid_source(p.parents[len(anchor.parts)-1])})
    candidates=folders or [p for p in sorted(search_root.rglob('*.zip')) if valid_source(p)]
    if len(candidates)!=1:raise ValueError(f'Need exactly one current code source; found{len(candidates)}: {candidates}. Set CODE_INPUT explicitly.')
    return candidates[0]


def extract_safe(archive,destination):
    destination=Path(destination).resolve();destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        seen=set()
        for item in z.infolist():
            name=PurePosixPath(item.filename.replace('\\','/'))
            if name.is_absolute() or '..' in name.parts or ':' in str(name) or stat.S_IFMT(item.external_attr>>16)==stat.S_IFLNK:
                raise ValueError('Unsafe archive member')
            key=str(name).rstrip('/').casefold()
            if key in seen:raise ValueError('Duplicate archive member')
            seen.add(key)
            if not (destination/str(name)).resolve().is_relative_to(destination):raise ValueError('Unsafe archive destination')
        z.extractall(destination)


def copy_source(source,destination):
    source=Path(source);destination=Path(destination)
    if not valid_source(source):raise ValueError('Stale or missing suite code')
    if source.is_file():extract_safe(source,destination)
    else:
        destination.mkdir(parents=True,exist_ok=True)
        for name in ('src','scripts','configs','tests','third_party'):
            if (source/name).exists():shutil.copytree(source/name,destination/name,dirs_exist_ok=True)
        for name in ('requirements.txt','pyproject.toml','THIRD_PARTY_NOTICES.md'):
            if (source/name).is_file():shutil.copy2(source/name,destination/name)

import os,sys,json,hashlib,time
STARTED=time.monotonic()
MODEL='deeplabv3plus_resnet50'
# Only input paths need configuring. Training policy is frozen in the suite.
CODE_INPUT=Path('/kaggle/input')
CITYSCAPES_ROOT=None
RESUME_INPUT=None  # This model's automatic suite ZIP/extracted workspace, when continuing.


def fetch_deeplab_release():
    from urllib.request import Request, urlopen
    import hashlib
    base='https://github.com/MinhToan1001/adverse-weather-segmentation/releases/download/deeplab-dg-seed42-resume-e4/'
    expected_manifest_sha256='484fbb195c6c07accea93abe7734ad9fec3a72ae20b70370b88345f11d363755'
    def request(url):
        return urlopen(Request(url,headers={'User-Agent':'Project-CV-checkpoint-resume'}),timeout=120)
    with request(base+'manifest.json') as response: raw=response.read()
    assert hashlib.sha256(raw).hexdigest()==expected_manifest_sha256, 'Release manifest hash mismatch'
    info=json.loads(raw)
    assert info['schema']=='deeplab_release_parts_v1' and info['model']=='deeplabv3plus_resnet50' and info['seed']==42
    folder=Path('/tmp/project_cv_deeplab_release');folder.mkdir(parents=True,exist_ok=True)
    target=folder/'deeplab_dg_resume_e4.zip'
    if target.is_file() and target.stat().st_size==info['archive']['bytes']:
        with target.open('rb') as handle:
            if hashlib.file_digest(handle,'sha256').hexdigest()==info['archive']['sha256']: return target
    partial=folder/'deeplab_dg_resume_e4.partial.zip'
    whole=hashlib.sha256();total=0
    with partial.open('wb') as writer:
        for part in info['parts']:
            name=part['name']
            assert Path(name).name==name and name.startswith('deeplab_dg_resume_e4.zip.part')
            digest=hashlib.sha256();size=0;next_report=128*1024*1024
            print('Downloading checkpoint:',name,flush=True)
            with request(base+name) as response:
                while True:
                    chunk=response.read(8*1024*1024)
                    if not chunk: break
                    writer.write(chunk);digest.update(chunk);whole.update(chunk);size+=len(chunk);total+=len(chunk)
                    if size>=next_report:
                        print(name,round(100*size/part['bytes'],1),'%',flush=True)
                        next_report=size+128*1024*1024
            assert size==part['bytes'] and digest.hexdigest()==part['sha256'], 'Checkpoint part incomplete: '+name
    assert total==info['archive']['bytes'] and whole.hexdigest()==info['archive']['sha256'], 'Assembled checkpoint hash mismatch'
    os.replace(partial,target)
    return target

# Resume only: input result or the pinned public Release; never start a new run.
def find_deeplab_result(input_root):
    def matches(value):
        return (value.get('schema')=='automatic_dg_model_suite_v1'
            and value.get('protocol')=='cityscapes_only_dg_v1'
            and value.get('model')=='deeplabv3plus_resnet50' and value.get('seed')==42)
    folders=[]
    for file in Path(input_root).rglob('suite_plan.json'):
        try:
            if matches(json.loads(file.read_text())): folders.append(file.parent)
        except (ValueError,OSError): continue
    candidates=sorted(set(folders))
    if not candidates:
        for file in Path(input_root).rglob('*.zip'):
            try:
                with zipfile.ZipFile(file) as z:
                    if 'suite_plan.json' in z.namelist() and matches(json.loads(z.read('suite_plan.json'))):
                        candidates.append(file)
            except (ValueError,OSError,zipfile.BadZipFile): continue
    if not candidates:
        print('No DeepLab result input: fetching the verified GitHub Release checkpoint.',flush=True)
        return fetch_deeplab_release()
    assert len(candidates)==1, f'Need one DeepLab result input; found {candidates}. Set RESUME_INPUT explicitly.'
    return candidates[0]

if RESUME_INPUT is None:
    RESUME_INPUT=find_deeplab_result(Path('/kaggle/input'))
print('Resume existing DeepLab result:',RESUME_INPUT)

BASE=Path('/kaggle/working')
WORK=BASE/f'dg_{MODEL}_suite_seed42'
REPO=BASE/f'project_{MODEL}_suite'
ARCHIVE=BASE/f'dg_{MODEL}_suite_seed42.zip'
assert not WORK.exists() and not REPO.exists(),'Fresh session required; preserve previous result first'
if RESUME_INPUT is None:
    copy_source(resolve_source(CODE_INPUT,Path('/kaggle/input')),REPO)
else:
    previous=Path(RESUME_INPUT)
    if previous.is_file():extract_safe(previous,WORK)
    else:
        found=[previous] if (previous/'suite_plan.json').is_file() else [p.parent for p in previous.rglob('suite_plan.json')]
        assert len(found)==1,f'Set RESUME_INPUT to one automatic model suite: {found}'
        shutil.copytree(found[0],WORK)
    plan=json.loads((WORK/'suite_plan.json').read_text())
    assert plan['schema']=='automatic_dg_model_suite_v1' and plan['protocol']=='cityscapes_only_dg_v1'
    assert plan['model']==MODEL and plan['seed']==42,'Result belongs to another model/seed'
    copy_source(WORK/'source.zip' if (WORK/'source.zip').is_file() else WORK/'source',REPO)
    assert all(hashlib.sha256((REPO/n).read_bytes()).hexdigest()==h for n,h in plan['source_hashes'].items())
    if not (WORK/'source.zip').is_file():
        with zipfile.ZipFile(WORK/'source.zip','w',zipfile.ZIP_DEFLATED) as z:
            for name in plan['source_hashes']:z.write(REPO/name,name)
            for name in ('requirements.txt','THIRD_PARTY_NOTICES.md','third_party/transformers/LICENSE'):
                if (REPO/name).is_file():z.write(REPO/name,name)
    CITYSCAPES_ROOT=Path(plan['city_root'])
assert (REPO/'scripts/run_dg_suite.py').is_file(),'Upload the NEW automatic-suite project-code.zip'
if CITYSCAPES_ROOT is None:
    found=sorted({p.parent for p in Path('/kaggle/input').rglob('leftImg8bit') if p.is_dir()
                  and all((p.parent/n).is_dir() for n in ['leftImg8bit/train','gtFine/train','leftImg8bit/val','gtFine/val'])})
    assert len(found)==1,f'Set CITYSCAPES_ROOT explicitly: {found}'
    CITYSCAPES_ROOT=found[0]
os.chdir(REPO);sys.path.insert(0,str(REPO))
os.environ.update(USE_TF='0',USE_TORCH='1',OMP_NUM_THREADS='2',MKL_NUM_THREADS='2')
print('Automatic DG suite:',MODEL,'Cityscapes:',CITYSCAPES_ROOT)

# Complete arms must be retained; the remaining arm continues from last/.
for arm in ['baseline','aug']:
    last=WORK/'jobs'/arm/'run/last/checkpoint.json'
    assert last.is_file() and json.loads(last.read_text())['epoch']==20, f'{arm} must already have20epochs'
    assert (WORK/'jobs'/arm/'frozen_candidate.json').is_file()
last=WORK/'jobs/consistency/run/last/checkpoint.json'
assert last.is_file(), 'Use the DeepLab output ZIP with consistency checkpoint epoch4'
saved=int(json.loads(last.read_text())['epoch'])
assert 4 <= saved < 20, f'Consistency saved epoch={saved}; a completed suite needs no more training'
print(f'Baseline and aug complete. Continue consistency from epoch{saved+1} through20.')


In [ ]:
import subprocess
def run(*args):subprocess.run([sys.executable,'-u',*map(str,args)],check=True,cwd=REPO)
run('-m','pip','install','transformers==4.57.6','numpy>=1.26,<3','Pillow==11.3.0',
    'PyYAML>=6,<7','matplotlib>=3.8,<4','safetensors>=0.4')
import torch
assert torch.cuda.is_available(),'Enable GPU'
assert tuple(map(int,torch.__version__.split('+')[0].split('.')[:2]))>=(2,6),'Require Torch>=2.6'
print(torch.__version__,torch.cuda.get_device_name(0))


In [ ]:
# Public checkpoint excludes Cityscapes GT data. Recreate it from your input.
metadata=WORK/'release_handoff.json'
if metadata.is_file():
    import hashlib,numpy as np
    from PIL import Image
    from concurrent.futures import ThreadPoolExecutor
    from src.data.preparation import convert_label_ids
    marker=json.loads(metadata.read_text())
    assert marker['schema']=='deeplab_resume_without_dataset_v1'
    masks=json.loads((WORK/'omitted_cityscapes_masks.json').read_text())
    assert masks['schema']=='cityscapes_trainids_restore_v1' and len(masks['masks'])==3475
    city=Path(plan['city_root'])
    def restore_mask(item):
        relative=Path(item['path'])
        assert not relative.is_absolute() and '..' not in relative.parts and relative.parts[:2]==('data','converted')
        destination=WORK/relative
        if not destination.is_file():
            native=city/relative.relative_to(Path('data/converted'))
            raw=native.with_name(native.name.replace('_labelTrainIds.png','_labelIds.png'))
            destination.parent.mkdir(parents=True,exist_ok=True)
            if native.is_file(): shutil.copy2(native,destination)
            else:
                assert raw.is_file(), f'Cityscapes GT missing: {raw}. Attach the same complete Cityscapes input.'
                convert_label_ids(raw,destination)
        with Image.open(destination) as image: pixels=np.asarray(image,dtype=np.uint8)
        assert list(pixels.shape)==item['shape'] and hashlib.sha256(pixels.tobytes()).hexdigest()==item['pixel_sha256'], f'Converted GT differs from original run: {destination}'
    print('Restoring3475Cityscapes masks from your own input...',flush=True)
    with ThreadPoolExecutor(max_workers=4) as pool: list(pool.map(restore_mask,masks['masks']))
    print('All3475mask pixel hashes match the original run. Frozen config and manifests unchanged.')


In [ ]:
from src.experiments.packaging import export_workspace
from IPython.display import FileLink,display
remaining=10-(time.monotonic()-STARTED)/3600
try:
    if remaining>0:
        run('-m','scripts.run_dg_suite','--city',CITYSCAPES_ROOT,'--model',MODEL,
            '--root',WORK,'--archive',ARCHIVE,'--session-hours',remaining)
    else:print('Setup used this session window. Preserve inputs and start a fresh session.')
finally:
    # The runner already exports after each epoch and on Python failures.
    # Avoid compressing all checkpoints twice on a normal return.
    if WORK.exists() and not ARCHIVE.is_file():export_workspace(WORK,ARCHIVE)
    if ARCHIVE.is_file():print('Download:',ARCHIVE);display(FileLink(str(ARCHIVE)))


In [ ]:
if (WORK/'suite_status.json').is_file():
    status=json.loads((WORK/'suite_status.json').read_text())
    print('Suite:',status['status'])
    for job in status.get('jobs',[]):
        print(job['arm'],str(job['completed_epochs'])+'/20',job['status'],
              'Cityscapes val mIoU:',job.get('cityscapes_val_miou'))
    if status['status']=='session_checkpoint_ready':
        print('Preserve this ZIP. Next fresh session: set only RESUME_INPUT; completed epochs/profile are reused.')


suite_complete nghĩa là đủ3×20epoch của model này. Sau cả hai suite, khóa6candidate rồi mới evaluate ACDC. Điểm trong bảng là Cityscapes val dùng selection, không phải kết quả ACDC. Tải ZIP suite trước khi xóa runtime; mất giữa epoch sẽ chạy lại epoch đó.